In [1]:
# Colab 사전 설치 & 버전 확인 (2025 기준)
# - PyTorch 2.2+ / Transformers 4.45+ / Datasets 3.0+ 권장
# - 이미 설치되어 있어도 최신으로 맞춤

!pip -q install -U "torch>=2.2, <3.0" "torchvision>=0.17, <1.0" "torchaudio>=2.2, <3.0"
!pip -q install -U "datasets>=3.0.1" "transformers>=4.45.2" "accelerate>=1.0.1" "evaluate>=0.4.2" "scikit-learn>=1.5.2"

import torch, transformers, datasets, sklearn, evaluate, sys, platform
print("Python            :", sys.version.split()[0])
print("Platform          :", platform.platform())
print("PyTorch           :", torch.__version__)
print("Transformers      :", transformers.__version__)
print("Datasets          :", datasets.__version__)
print("scikit-learn      :", sklearn.__version__)
print("evaluate          :", evaluate.__version__)
print("CUDA available    :", torch.cuda.is_available())
print("GPU device        :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 142.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.

In [2]:
# 준비: 라이브러리 & 시드
import random, numpy as np, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
import evaluate

SEED = 2025
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

In [3]:
dataset = load_dataset("stanfordnlp/imdb")  # train/test
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [4]:
# 토크나이저/모델 불러오기
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
# 전처리 함수: 토큰화
MAX_LEN = 256

def preprocess(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

encoded = dataset.map(preprocess, batched=True, remove_columns=["text"])
print(encoded)
# 토크나이즈 한 후에 원본 텍스트는 제거 >> 메모리 절약

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})


In [ ]:
# 데이터 콜레이터(동적 패딩)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# 평가지표: 정확도/정밀도/재현율/F1
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")
prec_metric= evaluate.load("precision")
rec_metric = evaluate.load("recall")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
        "precision": prec_metric.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall": rec_metric.compute(predictions=preds, references=labels, average="binary")["recall"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

In [ ]:
# 학습 설정(Trainer)
# 드롭인 패치: TrainingArguments 버전 호환 생성기
from transformers import TrainingArguments
from inspect import signature
import torch

def make_training_args(**base):
    """
    TrainingArguments의 인자 지원 여부를 자동 감지하여
    호환 가능한 키만 주입해 생성합니다.
    """
    sig = signature(TrainingArguments.__init__).parameters
    # 인자 지원 여부 확인
    print("인자 지원 여부:",list(sig.keys())[:10])
    args = {}

    # 공통/안전한 기본값 설정해 줌
    defaults = dict(
        output_dir="/content/imdb_distilbert_2025",
        per_device_train_batch_size=4,
        per_device_eval_batch_size=8,
        num_train_epochs=2,
        learning_rate=2e-5,
        weight_decay=0.01,
        logging_steps=100,
        seed=2025,
        report_to="none", # wandb 나 tensorboard
        fp16=torch.cuda.is_available(),
        disable_tqdm=True,

    )
    # fp16 (half-precision) 학습 여부(gpu 지원시, 메모리 아끼고 학습속도 2배 높여주는 방식)
    # 학습 데이터나 평가 데이터의 양이 많을 때, tqdm이 매 스텝마다 화면을 갱신하려고 시도하면 브라우저에 너무 많은 데이터를 보내기 때문
    defaults.update(base or {})

    # 지원되는 키만 선별 주입
    for k, v in defaults.items():
        if k in sig:
            args[k] = v

    # 평가/저장 전략 키 호환(evaluation_strategy / eval_strategy, save_strategy / save_steps)
    # 버전관리 (Transformers)
    if "evaluation_strategy" in sig:
        args["evaluation_strategy"] = defaults.get("evaluation_strategy", "epoch") # v4
    elif "eval_strategy" in sig:
        args["eval_strategy"] = defaults.get("evaluation_strategy", "epoch")       # v5

    if "save_strategy" in sig:
        args["save_strategy"] = defaults.get("save_strategy", "epoch")
    elif "save_steps" in sig:
        # epoch 단위 저장을 직접 지원 안하면, 간격(step)으로 대체
        # (여기선 대략적인 값으로 500 스텝)
        args["save_steps"] = defaults.get("save_steps", 500)

    # 베스트 모델 로드/모델 선택 지표(지원 시에만)
    if "load_best_model_at_end" in sig:
        args["load_best_model_at_end"] = defaults.get("load_best_model_at_end", True)
    if "metric_for_best_model" in sig:
        args["metric_for_best_model"] = defaults.get("metric_for_best_model", "accuracy")
    if "greater_is_better" in sig:
        args["greater_is_better"] = defaults.get("greater_is_better", True)

    return TrainingArguments(**args)

# 사용 예시(기존 TrainingArguments(...) 대신 아래 한 줄로 교체)
training_args = make_training_args(
    evaluation_strategy="epoch",   # 지원되면 사용, 아니면 자동 대체
    save_strategy="epoch"          # 지원되면 사용, 아니면 save_steps로 대체
)


인자 지원 여부: ['self', 'output_dir', 'per_device_train_batch_size', 'num_train_epochs', 'max_steps', 'learning_rate', 'lr_scheduler_type', 'lr_scheduler_kwargs', 'warmup_steps', 'optim']


In [ ]:
# Trainer 구성 & 학습
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["test"],
    # tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class = tokenizer
)

trainer.train()

eval_res = trainer.evaluate()
print("평가 결과:", eval_res)



{'loss': '0.1993', 'grad_norm': '0.02622', 'learning_rate': '1.984e-05', 'epoch': '0.016'}
{'loss': '0.2612', 'grad_norm': '0.4619', 'learning_rate': '1.968e-05', 'epoch': '0.032'}
{'loss': '0.1918', 'grad_norm': '0.01159', 'learning_rate': '1.952e-05', 'epoch': '0.048'}
{'loss': '0.2221', 'grad_norm': '0.02999', 'learning_rate': '1.936e-05', 'epoch': '0.064'}
{'loss': '0.1527', 'grad_norm': '159.4', 'learning_rate': '1.92e-05', 'epoch': '0.08'}
{'loss': '0.1864', 'grad_norm': '0.03061', 'learning_rate': '1.904e-05', 'epoch': '0.096'}
{'loss': '0.2382', 'grad_norm': '65.81', 'learning_rate': '1.888e-05', 'epoch': '0.112'}
{'loss': '0.1564', 'grad_norm': '0.02894', 'learning_rate': '1.872e-05', 'epoch': '0.128'}
{'loss': '0.2653', 'grad_norm': '31.38', 'learning_rate': '1.856e-05', 'epoch': '0.144'}
{'loss': '0.1333', 'grad_norm': '0.06185', 'learning_rate': '1.84e-05', 'epoch': '0.16'}
{'loss': '0.1204', 'grad_norm': '0.02318', 'learning_rate': '1.824e-05', 'epoch': '0.176'}
{'loss': '

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.121', 'grad_norm': '0.006934', 'learning_rate': '9.922e-06', 'epoch': '1.008'}
{'loss': '0.07371', 'grad_norm': '0.006969', 'learning_rate': '9.762e-06', 'epoch': '1.024'}
{'loss': '0.05422', 'grad_norm': '0.02025', 'learning_rate': '9.602e-06', 'epoch': '1.04'}
{'loss': '0.08113', 'grad_norm': '0.01265', 'learning_rate': '9.442e-06', 'epoch': '1.056'}
{'loss': '0.04247', 'grad_norm': '0.01186', 'learning_rate': '9.282e-06', 'epoch': '1.072'}
{'loss': '0.0856', 'grad_norm': '0.008564', 'learning_rate': '9.122e-06', 'epoch': '1.088'}
{'loss': '0.03692', 'grad_norm': '0.003688', 'learning_rate': '8.962e-06', 'epoch': '1.104'}
{'loss': '0.1149', 'grad_norm': '34.79', 'learning_rate': '8.802e-06', 'epoch': '1.12'}
{'loss': '0.08191', 'grad_norm': '0.01902', 'learning_rate': '8.642e-06', 'epoch': '1.136'}
{'loss': '0.06355', 'grad_norm': '0.005195', 'learning_rate': '8.482e-06', 'epoch': '1.152'}
{'loss': '0.03775', 'grad_norm': '115.5', 'learning_rate': '8.322e-06', 'epoch': '1

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'train_runtime': '531.7', 'train_samples_per_second': '94.04', 'train_steps_per_second': '23.51', 'train_loss': '0.1234', 'epoch': '2'}


In [ ]:
from transformers import Trainer

eval_pred = trainer.predict(encoded['test'])
print(eval_pred.metrics)

{'test_loss': 0.39594191312789917, 'test_accuracy': 0.91432, 'test_precision': 0.9185388718280265, 'test_recall': 0.90928, 'test_f1': 0.9138859853662459, 'test_runtime': 36.1643, 'test_samples_per_second': 691.289, 'test_steps_per_second': 86.411}


In [ ]:
# 추론
texts = [
    "This movie was absolutely wonderful and touching!",
    "I really hated this film. The acting was terrible."
]
inputs = tokenizer(texts, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN).to(trainer.model.device)
with torch.no_grad():
    probs = torch.softmax(trainer.model(**inputs).logits, dim=-1).cpu().numpy()

for t, p in zip(texts, probs):
    print(f"문장: {t}\n  → Negative={p[0]:.3f}, Positive={p[1]:.3f}")

문장: This movie was absolutely wonderful and touching!
  → Negative=0.001, Positive=0.999
문장: I really hated this film. The acting was terrible.
  → Negative=0.999, Positive=0.001
